In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [ ]:
!pip install datasets
from datasets import load_dataset

# Load dataset from Hugging Face
dataset = load_dataset("batterydata/pos_tagging")

# Display a sample sentence
print(dataset["train"][0])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/587 [00:00<?, ?B/s]

train.json:   0%|          | 0.00/5.05M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/601k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13054 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1451 [00:00<?, ? examples/s]

{'words': ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], 'labels': ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NN', 'NNS', 'IN', 'NNP', ',', 'JJ', 'IN', 'NN', 'NN', ',', 'VB', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NNP', 'CC', 'NNP', 'POS', 'JJ', 'NNS', '.']}


In [ ]:
from gensim.models import Word2Vec

# Extract sentences
sentences = [item["words"] for item in dataset["train"]]

# Train a Word2Vec model
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Example: Get vector for "battery"
print(w2v_model.wv["battery"][:5])  # Show first 5 values of embedding

[-0.01229496  0.01959288  0.01408443 -0.00853987 -0.01394627]


In [ ]:
# Extract sentences from the training dataset
sentences = [item["words"] for item in dataset["train"]]


def prepare_data(data, model):
    X, y = [], []
    for item in data:
        for word, pos in zip(item['words'][:1000], item['labels'][:1000]):
            if word in model.wv:
                X.append(model.wv[word])  # Get word vector
                y.append(pos)             # Corresponding POS tag
    return np.array(X), np.array(y)

# Prepare train and test data
X_train, y_train = prepare_data(dataset["train"], w2v_model)
X_test, y_test = prepare_data(dataset["test"], w2v_model)

# Encode POS tags as numerical labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Train Logistic Regression model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train_encoded)

# Predict and evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test_encoded, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("Predicted POS tags:", label_encoder.inverse_transform(y_pred))

Accuracy: 0.7573
Predicted POS tags: ['CC' 'NN' 'DT' ... 'JJ' 'NN' '.']


### Conclusion
As we can see here we have gotten an accuracy of 75% using a logistic regressions which we can improve by using more than the first 1000 lines of the training dataset or by using feature extraction.